# 07o — Literature comparator suite

This stage compares the frozen CP2-v2 semantic result with several published/open methodological baselines while keeping the upstream representation fixed.

```text
frozen 5,821 CP1 stays
        ↓
same per-stay IANA local time
        ↓
same complete-link 200 m production locations
        ↓
literature comparators
   ├─ scikit-mobility 1.3.1 HOME rule
   ├─ arXiv:2302.14742 hierarchical geohash HOME
   ├─ Pavan et al. area / intensity / frequency features
   └─ SCITEPRESS work-rest HOME/WORK ranker
        ↓
agreement / coverage / feature diagnostics
```

Important boundaries:

- This is **not** ground-truth evaluation.
- scikit-mobility is implemented as a source-equivalent replica of v1.3.1 `home_location`, avoiding an old runtime dependency while preserving the documented rule.
- The geohash paper is defined on raw monthly sightings; here it is explicitly adapted to one observation per occupied **stay-hour**.
- Pavan's accessible paper abstract defines the three feature dimensions but does not provide a public reproducible classifier formula, so this stage audits the feature space only and does **not** invent an importance score.
- The SCITEPRESS paper demonstrates one GeoLife user over seven days; here its top-two-region + work/rest-time idea is generalized across users as a style comparator.
- No result changes frozen HOME/OFFICE policy.

In [ ]:
from pathlib import Path
import importlib.util
import os
import subprocess
import sys

import numpy as np
import pandas as pd

REPO_URL='https://github.com/tanh1c/geolife.git'
BRANCH=os.environ.get('GEOLIFE_REPO_BRANCH','main')
REPO_DIR=Path(os.environ.get('GEOLIFE_REPO_DIR','/root/geolife'))
V=Path('/mnt/geolife-data')

def run(cmd,cwd=None):
    subprocess.run(cmd,cwd=cwd,check=True)

if not (REPO_DIR/'.git').exists():
    run(['git','clone','--branch',BRANCH,'--single-branch',REPO_URL,str(REPO_DIR)])
else:
    run(['git','-C',str(REPO_DIR),'fetch','origin',BRANCH])
    run(['git','-C',str(REPO_DIR),'checkout',BRANCH])
    run(['git','-C',str(REPO_DIR),'reset','--hard',f'origin/{BRANCH}'])

run([sys.executable,'-m','pip','install','-q','-e','.[dev]'],cwd=REPO_DIR)

for p in (str(REPO_DIR),str(REPO_DIR/'src')):
    if p not in sys.path:
        sys.path.insert(0,p)

from geolife.model import HomeOfficeConfig, build_semantic_locations, infer_home_office

spec=importlib.util.spec_from_file_location(
    's07o',
    REPO_DIR/'analysis'/'07o_literature_comparator_suite.py',
)
assert spec is not None and spec.loader is not None
s07o=importlib.util.module_from_spec(spec)
sys.modules[spec.name]=s07o
spec.loader.exec_module(s07o)

print('sha:',subprocess.run(
    ['git','-C',str(REPO_DIR),'rev-parse','--short','HEAD'],
    capture_output=True,text=True,check=True
).stdout.strip())
print(s07o.synthetic_self_check())

## 1. Frozen CP2-v2 parity gate

The entire suite uses the same frozen CP1 stays and production location namespace.

Expected:

- 5,821 stays / 136 users;
- 2,015 semantic locations;
- 716 recurring locations / 104 users;
- HOME 27;
- OFFICE 16.

In [ ]:
C=V/'cache'/'cp2_v2'/'07o_literature_comparator_suite'
C.mkdir(parents=True,exist_ok=True)

def find_cache(name):
    candidates=[
        V/'cache'/'03a_user_behavior_deep_dive'/name,
        V/'artifacts'/'03a'/name,
        REPO_DIR/'artifacts'/'03a'/name,
        V/'cache'/'cp2_home_office'/name,
    ]
    for p in candidates:
        if p.exists():
            return p
    hits=sorted(V.glob(f'**/{name}'))
    if hits:
        return hits[0]
    raise FileNotFoundError(name)

STAYS_PATH=find_cache('stays_baseline_v1.pkl')
stays=pd.read_pickle(STAYS_PATH)

cfg=HomeOfficeConfig()
semantic_stays,locations=build_semantic_locations(stays,config=cfg)
production=infer_home_office(stays,config=cfg)

counts=production['label'].value_counts().to_dict()
recurring=locations.loc[locations['stay_count'].ge(2)]

assert len(stays)==5821
assert stays['user_id'].astype(str).nunique()==136
assert len(semantic_stays)==5821
assert len(locations)==2015
assert len(recurring)==716
assert recurring['user_id'].astype(str).nunique()==104
assert counts.get('HOME',0)==27
assert counts.get('OFFICE',0)==16

parity=pd.DataFrame([{
    'cp1_stays':len(stays),
    'stay_users':stays['user_id'].astype(str).nunique(),
    'semantic_locations':len(locations),
    'recurring_locations':len(recurring),
    'recurring_location_users':recurring['user_id'].astype(str).nunique(),
    'HOME':counts.get('HOME',0),
    'OFFICE':counts.get('OFFICE',0),
}])
print('frozen CP2-v2 parity: PASS')
display(parity)

## 2. Shared literature observation adapters

Two frozen views are materialized:

1. **stay arrivals** — one observation per stay at the stable production location center;
2. **stay-hour occupancy** — one observation for every local hour touched by a stay.

The first is used for the scikit-mobility source-equivalent visit-count rule.

The second is used only where a paper expects repeated timestamped sightings / time occupancy.

In [ ]:
arrival_obs=s07o.build_stay_arrival_observations(semantic_stays,locations)
hour_obs=s07o.build_stay_hour_observations(semantic_stays,locations)

assert len(arrival_obs)==5821
assert arrival_obs['user_id'].nunique()==136
assert hour_obs['user_id'].nunique()==136

adapter_summary=pd.DataFrame([{
    'arrival_observations':len(arrival_obs),
    'stay_hour_observations':len(hour_obs),
    'arrival_users':arrival_obs['user_id'].nunique(),
    'stay_hour_users':hour_obs['user_id'].nunique(),
}])
display(adapter_summary)

## 3. scikit-mobility 1.3.1 HOME comparator

The v1.3.1 source defines `home_location` as the exact latitude/longitude with the most observations during **22:00–07:00**. If a user has no nighttime observation, it falls back to the most frequently observed location overall.

Stage 07o executes a source-equivalent implementation on frozen stay-arrival observations. Location coordinates are production location centers, so an exact location-id comparison is valid.

This is **HOME only**; scikit-mobility 1.3.1 does not expose an analogous WORK measure in its individual-measures API.

In [ ]:
scikit_home=s07o.scikit_mobility_home_replica(
    arrival_obs,
    start_night='22:00',
    end_night='07:00',
)
scikit_comparison=s07o.exact_namespace_comparison(
    semantic_stays['user_id'].astype(str).unique(),
    production,
    scikit_home,
    method='SCIKIT_MOBILITY_1_3_1_HOME',
    labels=('HOME',),
)
scikit_summary=s07o.summarize_exact_comparison(scikit_comparison)

print('SCIKIT-MOBILITY HOME')
display(scikit_summary)
print('fallback users:',int(scikit_home['used_fallback_all_visits'].sum()))

## 4. arXiv:2302.14742 hierarchical geohash HOME

The paper's monthly algorithm:

```text
level-6 geohashes
→ keep cells observed >=3 days
→ keep cells observed > half of observed days
→ require >=2 distinct hours/day on average
→ top 3 by days / hours / sightings
→ rerank top 3 by nights / nighttime hours / nighttime sightings
→ HOME level-6
→ repeat inside that cell at level-7
```

Night is **21:00–06:00**.

The paper uses raw mobile-device sightings. GeoLife CP2 does not retain a raw-point cache, so Stage 07o predeclares a **stay-hour adaptation**: each local hour occupied by a frozen stay is one observation.

The paper is monthly. Because GeoLife spans years, the notebook runs the algorithm per user-month and then uses a project-specific cross-month aggregation: modal level-7 candidate, tie-broken by accumulated night hours. Monthly coverage and modal stability are reported so this extra aggregation is visible.

In [ ]:
geohash_monthly,geohash_audit=s07o.geohash2302_monthly_home(hour_obs)
geohash_home=s07o.aggregate_geohash2302_home(geohash_monthly)
geohash_comparison=s07o.spatial_home_comparison(
    semantic_stays['user_id'].astype(str).unique(),
    production,
    geohash_home,
)
geohash_summary=s07o.summarize_spatial_home(geohash_comparison)

geohash_coverage=pd.DataFrame([{
    'user_months_seen':len(geohash_audit),
    'user_months_with_home':len(geohash_monthly),
    'users_with_home':geohash_home['user_id'].nunique() if len(geohash_home) else 0,
    'median_qualifying_months':float(geohash_home['qualifying_months'].median()) if len(geohash_home) else np.nan,
    'median_modal_month_share':float(geohash_home['modal_month_share'].median()) if len(geohash_home) else np.nan,
}])

print('ARXIV2302 GEOHASH HOME COVERAGE')
display(geohash_coverage)
print('ARXIV2302 GEOHASH HOME SPATIAL AGREEMENT')
display(geohash_summary)

## 5. PAVAN — area / intensity / frequency feature-space audit

Pavan et al. (MDM 2015) evaluate important-location recognition on GeoLife and map candidate places into three dimensions:

- **area**;
- **intensity** = time spent;
- **frequency** = number of visits.

The accessible abstract does not provide enough detail to reproduce a unique classification formula. Therefore Stage 07o does **not** invent one.

Production locations already provide intensity and frequency directly. For area, this notebook uses an explicitly named proxy:

```text
area_proxy_m2 = π × (production diameter / 2)^2
```

This is a descriptive feature-space audit, not a replication of their classifier.

In [ ]:
pavan_private,pavan_summary,pavan_rank_summary=s07o.pavan_feature_space(
    locations,
    production,
)

print('PAVAN FEATURE SPACE BY PRODUCTION ROLE')
display(pavan_summary)

print('PAVAN LABEL RANK AUDIT')
display(pavan_rank_summary)

## 6. SCITEPRESS/MATEC 2018 work-rest comparator

The paper demonstrates GeoLife **user 159 over seven days**. It takes the two largest regions and compares time proportions:

- **00:00–06:00** → Home;
- **08:00–18:00** → workplace;
- 18:00–24:00 → leisure interpretation.

Stage 07o generalizes only the first two pieces:

```text
top 2 production locations by stay_count
→ choose max 00–06 dwell as HOME
→ choose max 08–18 dwell as WORK
```

No weekday restriction is added because the cited paper passage does not specify one. This is a **SCITEPRESS-style comparator**, not an exact reproduction of its ADPC clustering or Google reverse-geocoding pipeline.

In [ ]:
scitepress_candidates=s07o.scitepress_style_candidates(
    semantic_stays,
    locations,
)
scitepress_comparison=s07o.exact_namespace_comparison(
    semantic_stays['user_id'].astype(str).unique(),
    production,
    scitepress_candidates,
    method='SCITEPRESS_WORK_REST_STYLE',
    labels=('HOME','OFFICE'),
)
scitepress_summary=s07o.summarize_exact_comparison(scitepress_comparison)

collision_users=(
    scitepress_candidates
    .groupby('user_id')['same_home_work_candidate']
    .max()
    .sum()
)

print('SCITEPRESS-STYLE HOME/WORK')
display(scitepress_summary)
print('same HOME/WORK candidate users:',int(collision_users))

## 7. Stage-07j near-miss WORK diagnostic

If the private 07j panel is present, compare the SCITEPRESS-style 08–18 WORK candidate with the nine frozen one-step OFFICE near-misses.

This is diagnostic only and cannot reopen Stage 07l.

In [ ]:
NEAR_PATH=V/'cache'/'cp2_v2'/'07j_near_miss_office_audit'/'near_miss_office_audit_private.pkl'
near_private=pd.DataFrame()
near_summary=pd.DataFrame()

if NEAR_PATH.exists():
    near_panel=pd.read_pickle(NEAR_PATH)
    assert len(near_panel)==25
    assert int(near_panel['audit_group'].ne('baseline').sum())==9
    near_private,near_summary=s07o.near_miss_work_summary(
        near_panel,
        scitepress_candidates,
        method='SCITEPRESS_WORK_REST_STYLE',
    )
    print('SCITEPRESS WORK × STAGE-07j NEAR-MISS')
    display(near_summary)
else:
    print('Stage-07j private panel not found; optional diagnostic skipped:',NEAR_PATH)

## 8. Cross-comparator synthesis

The suite deliberately contains methods with different semantics, so only comparable quantities are placed together:

- exact candidate match when using the same production location namespace;
- spatial distance for the geohash method, whose cells define a different namespace;
- feature-space characterization for Pavan, with no artificial accuracy score.

In [ ]:
suite_rows=[]

for row in scikit_summary.itertuples(index=False):
    suite_rows.append({
        'comparator':row.method,
        'label':row.label,
        'coverage_users':int(row.comparator_selected),
        'production_reference':int(row.production_emitted),
        'agreement_definition':'exact location_id',
        'agreement_count':int(row.exact_matches),
        'agreement_rate_vs_production':float(row.exact_rate_production),
    })

for row in scitepress_summary.itertuples(index=False):
    suite_rows.append({
        'comparator':row.method,
        'label':row.label,
        'coverage_users':int(row.comparator_selected),
        'production_reference':int(row.production_emitted),
        'agreement_definition':'exact location_id',
        'agreement_count':int(row.exact_matches),
        'agreement_rate_vs_production':float(row.exact_rate_production),
    })

g=geohash_summary.iloc[0]
suite_rows.append({
    'comparator':'ARXIV2302_GEOHASH_STAY_HOUR',
    'label':'HOME',
    'coverage_users':int(g['comparator_selected']),
    'production_reference':int(g['production_HOME']),
    'agreement_definition':'within 200m',
    'agreement_count':int(g['within_200m']),
    'agreement_rate_vs_production':(
        float(g['within_200m'])/float(g['production_HOME'])
        if g['production_HOME'] else np.nan
    ),
})

suite_summary=pd.DataFrame(suite_rows)
display(suite_summary)

## 9. Save outputs

User-level candidate identities and monthly geohash details remain private on the Modal Volume. Aggregate tables are safe for documentation.

In [ ]:
# Private
arrival_obs.to_pickle(C/'stay_arrival_observations_private.pkl')
hour_obs.to_pickle(C/'stay_hour_observations_private.pkl')
scikit_home.to_pickle(C/'scikit_home_candidates_private.pkl')
scikit_comparison.to_pickle(C/'scikit_home_comparison_private.pkl')
geohash_monthly.to_pickle(C/'geohash_monthly_home_private.pkl')
geohash_audit.to_pickle(C/'geohash_monthly_audit_private.pkl')
geohash_home.to_pickle(C/'geohash_user_home_private.pkl')
geohash_comparison.to_pickle(C/'geohash_home_comparison_private.pkl')
pavan_private.to_pickle(C/'pavan_feature_space_private.pkl')
scitepress_candidates.to_pickle(C/'scitepress_candidates_private.pkl')
scitepress_comparison.to_pickle(C/'scitepress_comparison_private.pkl')
if len(near_private):
    near_private.to_pickle(C/'scitepress_near_miss_private.pkl')

# Aggregate
parity.to_csv(C/'production_parity.csv',index=False)
adapter_summary.to_csv(C/'adapter_summary.csv',index=False)
scikit_summary.to_csv(C/'scikit_home_summary.csv',index=False)
geohash_coverage.to_csv(C/'geohash_coverage_summary.csv',index=False)
geohash_summary.to_csv(C/'geohash_home_summary.csv',index=False)
pavan_summary.to_csv(C/'pavan_feature_summary.csv',index=False)
pavan_rank_summary.to_csv(C/'pavan_rank_summary.csv',index=False)
scitepress_summary.to_csv(C/'scitepress_summary.csv',index=False)
suite_summary.to_csv(C/'literature_suite_summary.csv',index=False)
if len(near_summary):
    near_summary.to_csv(C/'scitepress_near_miss_summary.csv',index=False)

print('saved:',C)

## Interpretation boundary

- **SCIKIT_MOBILITY_1_3_1_HOME** reproduces the published v1.3.1 source rule on a frozen stay-arrival adapter; it is not an independent raw-GPS pipeline.
- **ARXIV2302_GEOHASH_STAY_HOUR** preserves the paper's monthly geohash support/ranking logic but replaces raw sightings with occupied stay-hours and adds an explicit cross-month modal aggregation for GeoLife.
- **PAVAN** is a feature-space audit only. Area is an explicit production-diameter proxy; no classifier or importance score is invented.
- **SCITEPRESS_WORK_REST_STYLE** generalizes a one-user/seven-day demonstration to the frozen production location namespace; it does not reproduce ADPC or reverse geocoding.
- Exact/spatial agreement is methodological convergence, not accuracy.
- No comparator output changes HOME 27 / OFFICE 16 or promotes Stage-07j near-misses.